In [1]:
# Google Colab setup 
from google.colab import drive
drive.mount('/content/drive')

# Install imagehash library for image processing
!pip install imagehash

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 27.7 MB/s eta 0:00:00


In [2]:
# Import data from kaggle
import kagglehub

# Download latest version
data_path = kagglehub.dataset_download("kaborg15/vibrent-clothes-rental-dataset")

print("Path to dataset files:", data_path)

100%|██████████| 6.17G/6.17G [05:08<00:00, 21.5MB/s] 

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/kaborg15/vibrent-clothes-rental-dataset/versions/7


In [3]:
# Import libraries
import pandas as pd
import numpy as np
from PIL import Image
import imagehash
import os
from torchvision import models
import torch
import shutil

In [ ]:
# Load outfits and picture triplets csv and drop irrelevant columns
data_outfits = pd.read_csv(f'{data_path}/outfits.csv', sep=';')
data_outfits = data_outfits[['id','group']]
# Load picture triplets csv
data_picture = pd.read_csv(f'{data_path }/picture_triplets.csv', sep=';')

In [5]:
data_outfits.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15649 entries, 0 to 15648
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      15649 non-null  object
 1   group   15626 non-null  object
dtypes: object(2)
memory usage: 244.6+ KB


In [6]:
data_picture.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50193 entries, 0 to 50192
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   picture.id    50193 non-null  object
 1   outfit.id     50193 non-null  object
 2   displayOrder  50193 non-null  int64 
 3   file_name     50193 non-null  object
dtypes: int64(1), object(3)
memory usage: 1.5+ MB


In [7]:
data_outfits.dropna(inplace=True)
data_outfits.info()

<class 'pandas.core.frame.DataFrame'>
Index: 15626 entries, 0 to 15648
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      15626 non-null  object
 1   group   15626 non-null  object
dtypes: object(2)
memory usage: 366.2+ KB


In [8]:
# Check for outfits without corresponding pictures
data_picture_null = data_outfits[~data_outfits['id'].isin(data_picture['outfit.id'])]
len(data_picture_null)

480

In [9]:
# Check for pictures without corresponding outfits
data_outfits_null = data_picture[~data_picture['outfit.id'].isin(data_outfits['id'])]
len(data_outfits_null)

31

In [10]:
# Remove any records without corresponding pictures
data_outfits = data_outfits[data_outfits['id'].isin(data_picture['outfit.id'])]
data_outfits.info()

<class 'pandas.core.frame.DataFrame'>
Index: 15146 entries, 0 to 15648
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      15146 non-null  object
 1   group   15146 non-null  object
dtypes: object(2)
memory usage: 355.0+ KB


In [11]:
# Remove any pictures without corresponding outfits
data_picture = data_picture[data_picture['outfit.id'].isin(data_outfits['id'])]
data_picture.info()

<class 'pandas.core.frame.DataFrame'>
Index: 50162 entries, 0 to 50192
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   picture.id    50162 non-null  object
 1   outfit.id     50162 non-null  object
 2   displayOrder  50162 non-null  int64 
 3   file_name     50162 non-null  object
dtypes: int64(1), object(3)
memory usage: 1.9+ MB


In [12]:
print(f'Length of data_outfits: {len(data_outfits)}')
print(f'Length of data_picture: {len(data_picture)}')

Length of data_outfits: 15146
Length of data_picture: 50162


In [13]:
# Create image path and check if image exists
data_picture['image_path'] = data_picture['file_name'].apply(lambda x: f"{data_path}/images/{x}")
data_picture['image_exists'] = data_picture['image_path'].apply(lambda x: os.path.exists(x))
data_picture['image_exists'].value_counts()

,count
image_exists,
True,50162


In [14]:
# Merge both dataframes 
merged_data = data_picture.merge(data_outfits[['id', 'group']], left_on='outfit.id', right_on='id', how='left').drop(columns=['id', 'image_exists'])

print(f"Merged data shape: {merged_data.shape} \n" )
print(f"Merged data missing values: \n{merged_data.isna().sum()}")

Merged data shape: (50162, 6) 

Merged data missing values: 
picture.id      0
outfit.id       0
displayOrder    0
file_name       0
image_path      0
group           0
dtype: int64


In [15]:
# Define function to verify images are not corrupted and meet minimum size requirements
def verify_image(image):
    try:
        with Image.open(image) as img:
            img.verify()
            w, h = img.size
            return w >= 100 and h >= 100
        return True
    except Exception:
        return False
    
merged_data['valid_image'] = merged_data['image_path'].apply(lambda x: verify_image(x))

In [16]:
merged_data['valid_image'].value_counts()

,count
valid_image,
True,50148
False,14


In [17]:
# Remove invalid images
merged_data = merged_data[merged_data['valid_image']==True]
merged_data['valid_image'].value_counts()

,count
valid_image,
True,50148


In [18]:
# Define function to check for duplicate images using phash
def check_duplicates(image):
    return str(imagehash.phash(Image.open(image)))

merged_data['image_phash'] = merged_data['image_path'].apply(lambda x: check_duplicates(x))
print(f'Duplicated images: {merged_data["image_phash"].duplicated().sum()}')

Duplicated images: 19782


In [19]:
# Drop duplicated images and remove unnecessary columns
merged_data = merged_data.drop_duplicates(subset='image_phash', keep='first').reset_index(drop=True).drop(columns=['valid_image', 'image_phash'])

print(f'Remaining images after removing duplicates: {len(merged_data)}')

Remaining images after removing duplicates: 30366


In [20]:
merged_data.head()

,picture.id,outfit.id,displayOrder,file_name,image_path,group
0,picture.0000cdba64314d84a49ed1c266589cc0,outfit.794483397da8425a813301eecf9828c6,0,0000cdba64314d84a49ed1c266589cc0.jpg,/root/.cache/kagglehub/datasets/kaborg15/vibre...,group.0a045453ba8fda6cd089eb66930233fb
1,picture.00058abb53434872ae9bb4270ae21f8e,outfit.98f32aaf08bc4ff09c44e6e11e9199bc,2,00058abb53434872ae9bb4270ae21f8e.jpg,/root/.cache/kagglehub/datasets/kaborg15/vibre...,group.cd76a901746b1b6c34cb6ede1586461e
2,picture.00063f52c36d43ada95da45f819b30b4,outfit.9fd1c42c3db543c5b6e53b0db1ee8c0f,3,00063f52c36d43ada95da45f819b30b4.jpg,/root/.cache/kagglehub/datasets/kaborg15/vibre...,group.985203168fe1cc282e1da350315d9acc
3,picture.0008443461814f5c988f123718bbd20e,outfit.a7539783b6e94591bdf4e10339afc1d7,3,0008443461814f5c988f123718bbd20e.jpg,/root/.cache/kagglehub/datasets/kaborg15/vibre...,group.957dbc51df1f0cfa6f38cb99d7b1e52c
4,picture.000a5db3362049aebcc1eb2bf7bde95f,outfit.745fa2bc8156478bac6c0f7d46dadbda,1,000a5db3362049aebcc1eb2bf7bde95f.jpg,/root/.cache/kagglehub/datasets/kaborg15/vibre...,group.f34c702b93b819798d7827d8751c143e


In [21]:
model_name = 'mobilenet_v3_large'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Set model and weights
weights = models.MobileNet_V3_Large_Weights.DEFAULT
model = models.mobilenet_v3_large(weights=weights)

# Replace classification layer with identity to avoid computing class scores and retrieve embeddings
model.classifier = torch.nn.Identity()

# Move to device and set to inference mode
model = model.to(device).eval()

preprocessor = weights.transforms()  

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 184MB/s]


In [22]:
# Save path for Colab
save_path = '/content/drive/MyDrive/data'
# Save path for local
# save_path = '/data/embeddings/'

# Set empty list for embeddings, ID and invalid images
all_embeddings = []
all_ids = []
invalid = []
for _, row in merged_data.iterrows():
    try:
        image = Image.open(f'{row['image_path']}').convert('RGB')
    except Exception as e:
        print(f'Error with: {row["file_name"]}: {e}')
        invalid.append((row['picture.id'], row['file_name']))
        continue

    inputs = preprocessor(image).unsqueeze(0).to(device)
    with torch.no_grad():
        embeddings = model(inputs).squeeze().cpu().numpy()

    all_embeddings.append(embeddings)
    all_ids.append(row['picture.id'])

print(f'Total embeddings: {len(all_embeddings)}, Total IDs: {len(all_ids)}')
print(f'Total invalid images: {len(invalid)}')

np.save(f'{save_path}/embeddings/embeddings.npy', np.array(all_embeddings))
np.save(f'{save_path}/embeddings/picture_ids.npy', np.array(all_ids))

print(f'Embeddings saved to: {save_path}/embeddings/embeddings.npy')
print(f'Picture IDs saved to: {save_path}/embeddings/picture_ids.npy')

Total embeddings: 30366, Total IDs: 30366
Total invalid images: 0
Embeddings saved to: /content/drive/MyDrive/data/embeddings/embeddings.npy
Picture IDs saved to: /content/drive/MyDrive/data/embeddings/picture_ids.npy


In [23]:
# Create embeddings DF and merge it with merged_data
embeddings_df = pd.DataFrame({'picture.id': all_ids, 'embeddings_idx': range(len(all_embeddings))})
catalog = merged_data.merge(embeddings_df, on='picture.id', how='inner')

# Save DF as catalog.csv
print(f'Full_data: {len(merged_data)}')
print(f'Catalog: {len(catalog)}')
catalog.to_csv(f'{save_path}/catalog.csv', index=False)

Full_data: 30366
Catalog: 30366


In [ ]:
# RUN THIS ONLY IF YOU WANT TO COPY IMAGES TO THE LOCAL ENVIRONMENT
# WORKS ONLY FOR LOCAL ENVIRONMENT

for image in catalog['file_name']:
    old_image = f'{data_path}/images/{image}'
    shutil.copy(old_image, f'data/copied_images/{image}')
print("All images copied successfully.")
